In [1]:
from main import region_generator
from pydrake.all import (
    StartMeshcat,
    AddDefaultVisualization,
    Simulator,
    RobotDiagramBuilder,
    VPolytope,
    HPolyhedron,
    SceneGraphCollisionChecker,
    RandomGenerator,
    PointCloud,
    Rgba,
    Quaternion,
    RigidTransform,
    IrisFromCliqueCoverOptions,
    IrisInConfigurationSpaceFromCliqueCoverV2,
    SaveIrisRegionsYamlFile,
    LoadModelDirectives,
    ProcessModelDirectives,
    CollisionCheckerParams,
    Sphere,
    GaussianVectorX,
    RandomGenerator,
    UniformVector,
    IrisZoFromCliqueBuilder,
    MinCliqueCoverSolverViaGreedy,
    MaxCliqueSolverViaGreedy,
    Parallelism,
    PointsToCliqueCoverSets,
    VisibilityGraph,
)
import numpy as np
import os
import data.iris_benchmarks.benchmarks.helpers as benchmark_helpers
from data.iris_benchmarks.benchmarks.helpers import load_seed_points
from data.iris_benchmarks.iris_environments.environments import get_environment_builder
import networkx as nx
import tqdm

In [2]:
TEST_SCENE = "7DOFIIWA"

src_directory = os.path.abspath(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
parent_directory = os.path.dirname(src_directory)
data_directory = os.path.join(parent_directory, "data")

region_file = os.path.join(data_directory, "iris_regions" + TEST_SCENE + ".yaml")

plant, scene_graph, diagram, diagram_context, plant_context, models, meshcat =  get_environment_builder(TEST_SCENE)(True)
diagram.ForcedPublish(diagram_context)
print("Model Names:")
for i, model in enumerate(models):
    print(f"models[{i}]: {model.model_name}")

INFO:drake:Meshcat listening for connections at http://localhost:7000


Model Names:
models[0]: iiwa
models[1]: wsg
models[2]: shelves1
models[3]: shelves2
models[4]: ground


In [3]:
collision_checker = SceneGraphCollisionChecker(model = diagram,
                                               robot_model_instances = [model.model_instance for model in models],
                                               edge_step_size = 0.02)

INFO:drake:Allocating contexts to support implicit context parallelism 16


In [4]:
seed_points = benchmark_helpers.load_seed_points(TEST_SCENE)
def view_points_as_spheres(points, meshcat, name, radius=0.01, color = Rgba(1,0,0,1)):
    for i, point in enumerate(points):
        cur_name = f"{name}/{i}"
        meshcat.SetObject(cur_name, Sphere(radius), color)
        meshcat.SetTransform(cur_name, RigidTransform(p=point[:, np.newaxis]))

end_effector_model_info = models[1]
print("End Effector Bodies:")
for body_index in plant.GetBodyIndices(end_effector_model_info.model_instance):
    print(plant.get_body(body_index).name())
end_effector = plant.GetBodyByName("body", end_effector_model_info.model_instance)
end_effector_frame_id = plant.GetBodyFrameIdOrThrow(end_effector.index())
def get_end_effector_pose(q, context = None):
    if context is None:
        context = plant.CreateDefaultContext()
    plant.SetPositions(context, q)
    return plant.EvalBodyPoseInWorld(context, end_effector).translation()

def plot_end_effector_configurations(configs, meshcat, name, radius=0.05, color = Rgba(1,0,0,1)):
    context = plant.CreateDefaultContext()
    task_space_points = []
    for q in configs:
        task_space_points.append(get_end_effector_pose(q, context))
    view_points_as_spheres(task_space_points, meshcat, name, radius, color)
    
def view_points_as_point_cloud(points, meshcat, name, size=0.01, color=Rgba(1, 0, 0, 1)):
    cloud = PointCloud(points.shape[0])
    cloud.mutable_xyzs()[:] = points.T
    meshcat.SetObject(name, cloud, point_size=size, rgba=color)


def plot_end_effector_configurations_as_point_cloud(
    configs, meshcat, name, radius=0.01, color=Rgba(1, 0, 0, 1)
):
    context = plant.CreateDefaultContext()
    task_space_points = []
    for q in configs:
        task_space_points.append(get_end_effector_pose(q, context))
    view_points_as_point_cloud(np.array(task_space_points), meshcat, name, radius, color)

plot_end_effector_configurations(seed_points, meshcat, "seed_points")

End Effector Bodies:
body
left_finger
right_finger


In [5]:
import time
for seed_point in seed_points:
    plant.SetPositions(plant_context, seed_point)
    diagram.ForcedPublish(diagram_context)
    # time.sleep(0.5)

shelf_seed_point_indices = list(range(1,6))
shelf_seed_points = seed_points[shelf_seed_point_indices]
    

In [6]:
class SphereicalRRT:
    def __init__(self, collision_checker, random_generator):
        self.collision_checker = collision_checker
        self.dim = self.collision_checker.plant().num_positions()
        self.random_generator = random_generator
        self.gaussian = GaussianVectorX(np.zeros(self.dim), np.ones(self.dim))
        self.uniform = UniformVector(np.zeros(1), np.ones(1))

    def sample_uniformly_in_sphere(self, seed_point, sphere_radius):
        direction = self.gaussian.Sample(self.random_generator)
        direction /= np.linalg.norm(direction)
        r = sphere_radius * self.uniform.Sample(self.random_generator).item()
        return r ** (1 / self.dim) * direction + seed_point

    def find_closest_point_in_tree(self, tree, point):
        """
        TODO decide how we pass tree.
        """
        point_distances_pairs = [
            (q, self.collision_checker.ComputeConfigurationDistance(np.array(q), point))
            for q in tree
        ]
        return np.array(min(point_distances_pairs, key=lambda x: x[1])[0])

    def extend(self, start_point, end_point):
        """
        Extend as far from start_point towards end_point and return the configuration
        """
        edge_measure = self.collision_checker.MeasureEdgeCollisionFree(
            start_point, end_point
        )
        assert edge_measure.partially_free()
        return self.collision_checker.InterpolateBetweenConfigurations(
            start_point, end_point, edge_measure.alpha()
        )

    def add_next_point(self, next_direction, tree):
        closest_point = self.find_closest_point_in_tree(tree, next_direction)
        next_point = self.extend(closest_point, next_direction)
        closest_point_tuple = tuple(closest_point)
        next_point_tuple = tuple(next_point)
        tree.add_node(next_point_tuple)
        tree.add_edge(closest_point_tuple, next_point_tuple)

    def build(
        self, seed_point, num_points, radius, goal=None, sample_goal_fraction=0.1
    ):
        assert seed_point.shape == (self.dim,)
        if goal is not None:
            assert goal.shape == seed_point.shape
        tree = nx.Graph()
        tree.add_node(tuple(seed_point))
        
        # pbar = tqdm.tqdm(desc="Building RRT", total=num_points)
        ctr = 0
        while tree.number_of_nodes() < num_points:
            if goal is not None and np.random.rand() < sample_goal_fraction:
                next_direction = goal
            else:
                next_direction = self.sample_uniformly_in_sphere(seed_point, radius)
            self.add_next_point(next_direction, tree)
            # pbar.update(tree.number_of_nodes())
            ctr += 1
        # pbar.close()
        return tree



rrt_builder = SphereicalRRT(collision_checker, RandomGenerator(seed=0))


In [7]:
import matplotlib.pyplot as plt

prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']
def hex_to_rgb_0_1(hex_color):
    # Remove the '#' if present
    hex_color = hex_color.lstrip('#')
    
    # Convert hex to RGB
    rgb = tuple(int(hex_color[i:i+2], 16)/255. for i in (0, 2, 4))
    return rgb

In [8]:
rrt_points = {}
seed_point_colors = {}
for i, seed_point in tqdm.tqdm(enumerate(seed_points)):
    tree = rrt_builder.build(
        seed_point=seed_point,
        num_points=1000,
        radius=1,
        goal=None,
        sample_goal_fraction=0.1,
    )
    rrt_points[i] = np.array(tree.nodes)
    color = Rgba(*hex_to_rgb_0_1(colors[i]), 1)
    plot_end_effector_configurations_as_point_cloud(rrt_points[i], meshcat, f"seed_point_{i}/rrt_points", radius=0.01, color=color)
    plot_end_effector_configurations(seed_point[np.newaxis, :], meshcat, f"seed_point_{i}", radius=0.1, color=color)
    

10it [00:31,  3.11s/it]


In [35]:
seed_point_index = 6

set_builder = IrisZoFromCliqueBuilder(collision_checker)
max_clique_solver = MaxCliqueSolverViaGreedy()
min_clique_cover_solver = MinCliqueCoverSolverViaGreedy(max_clique_solver)
points = rrt_points[seed_point_index].T
# sets = PointsToCliqueCoverSets(points, collision_checker, min_clique_cover_solver, set_builder)


In [36]:
t0 = time.time()
visibility_graph = VisibilityGraph(collision_checker, points, Parallelism.Max())
t1 = time.time()
print(f"Visibilty Graph took {t1-t0} seconds")

Visibilty Graph took 184.61074566841125 seconds


In [37]:
min_clique_cover_solver.set_min_clique_size(20)
t0 = time.time()
clique_cover = min_clique_cover_solver.SolveMinCliqueCover(visibility_graph, True)
t1 = time.time()
print(f"Clique Cover took {t1-t0} seconds")
print(f"There are {len(clique_cover)} cliques")

Clique Cover took 0.05350899696350098 seconds
There are 6 cliques


In [38]:
for clique in clique_cover:
    print(len(clique))

250
53
37
62
44
35


In [39]:
sets = []
for clique_inds in clique_cover:
    clique = np.zeros((points.shape[0],len(clique_inds)))
    for i, ind in enumerate(clique_inds):
        clique[:,i] = points[:,ind]
    sets.append(set_builder.BuildRegion(clique))
    # break

INFO:drake:FastIris iteration 0
INFO:drake:FastIris iteration 1
INFO:drake:FastIris iter 2, iter limit 2
INFO:drake:FastIris iteration 0
INFO:drake:FastIris iteration 1
INFO:drake:FastIris iter 2, iter limit 2
INFO:drake:FastIris iteration 0
INFO:drake:FastIris iteration 1
INFO:drake:FastIris iter 2, iter limit 2
INFO:drake:FastIris iteration 0
INFO:drake:FastIris iteration 1
INFO:drake:FastIris iter 2, iter limit 2
INFO:drake:FastIris iteration 0
INFO:drake:FastIris iteration 1
INFO:drake:FastIris iter 2, iter limit 2
INFO:drake:FastIris iteration 0
INFO:drake:FastIris delta vol 0.018727188475717634, threshold 0.02


In [40]:
regions_dict = {f"set{i}": s for i, s in enumerate(sets)}
SaveIrisRegionsYamlFile(region_file, regions_dict)

In [41]:
def visualize_region(region, meshcat, name, num_samples = 1000,  radius=0.01, color=Rgba(1, 0, 0, 1)):
    context = plant.CreateDefaultContext()
    task_space_points = []
    generator = RandomGenerator(0)
    sample = region.ChebyshevCenter()
    for i in range(num_samples):
        sample = region.UniformSample(generator, sample, mixing_steps = 25)
        task_space_points.append(get_end_effector_pose(sample, context))
    view_points_as_point_cloud(np.array(task_space_points), meshcat, name, radius, color)
for idx, set in enumerate(sets):
    color = Rgba(*hex_to_rgb_0_1(colors[idx]), 1)
    visualize_region(sets[idx], meshcat, f"seed_point_{seed_point_index}/set_{idx}", color = color) 
    

In [ ]:
set_builder = IrisZoFromCliqueBuilder(collision_checker)
max_clique_solver = MaxCliqueSolverViaGreedy()
min_clique_cover_solver = MinCliqueCoverSolverViaGreedy(max_clique_solver)
min_clique_cover_solver.set_min_clique_size(20)
sets = []
for seed_point_index, points in rrt_points.items():
    print(f"Starting Seed Point {seed_point_index+1}/{len(rrt_points)}")
    t0 = time.time()
    visibility_graph = VisibilityGraph(collision_checker, points.T, Parallelism.Max())
    t1 = time.time()
    print(f"Visibilty Graph took {t1-t0} seconds")
    
    t0 = time.time()
    clique_cover = min_clique_cover_solver.SolveMinCliqueCover(visibility_graph, True)
    t1 = time.time()
    print(f"Clique Cover took {t1-t0} seconds")
    print(f"There are {len(clique_cover)} cliques")
    print("Clique sizes =")
    for clique in clique_cover:
        print(len(clique))
    new_sets = []
    for clique_inds in clique_cover:
        clique = np.zeros((points.shape[0],len(clique_inds)))
        for i, ind in enumerate(clique_inds):
            clique[:,i] = points[:,ind]
        new_sets.append(set_builder.BuildRegion(clique))
    sets.append(new_sets)

Starting Seed Point 1/10


In [ ]:
regions_dict = {f"tree_{i}_set_{j}": s for i, set_i in enumerate(sets) for j, s in set_i}
SaveIrisRegionsYamlFile(region_file, regions_dict)
for seed_point_index, set_i in enumerate(sets):
    for j, s in set_i:
        color = Rgba(*hex_to_rgb_0_1(colors[(seed_point_index+j)%len(colors)]), 1)
        visualize_region(s, meshcat, f"seed_point_{seed_point_index}/set_{j}", color = color) 